# essay evaluation workflow:
START -> (clarity of thoughts, Depth of analysis, language quality) -> all 3 gives 2 things (1/score ,2/ text feedback) -> go to final evaluation node. aikhane last 3 ta theke asha info nia summary generate krbe allm, then average kroe final output dibe - final average score -> END

aikhane 3 ta jinish lagbe - parallel workflows, structured output, reducer function

In [10]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score : int = Field(description="Score out of 10", ge=0,le=10)
    

In [5]:
structure_model = model.with_structured_output(EvaluationSchema)

In [6]:
essay = '''**The Rise of AI in Bangladesh**

Artificial Intelligence (AI) is rapidly transforming Bangladesh and creating new opportunities in various sectors. With the growth of digital technology, businesses, educational institutions, and government organizations are increasingly adopting AI-based solutions to improve efficiency and productivity.

In the business sector, AI is being used for customer service, data analysis, fraud detection, and automation. Many startups and technology companies are developing AI-powered applications to solve local challenges. The banking and financial sectors are also using AI to enhance security and detect suspicious transactions.

Education is another area where AI is making a significant impact. Students now have access to intelligent learning tools, online tutors, and personalized educational resources. Universities are introducing courses related to AI, machine learning, and data science to prepare the next generation of professionals.

The government of Bangladesh is promoting digital transformation through various initiatives under the vision of a Smart Bangladesh. These efforts encourage innovation and create opportunities for AI research and development.

Despite its benefits, AI also presents challenges such as job displacement, data privacy concerns, and the need for skilled professionals. Therefore, proper policies, education, and ethical guidelines are essential for the responsible use of AI.

In conclusion, the rise of AI in Bangladesh is opening new possibilities for economic growth, innovation, and social development. With continued investment in technology and education, AI can play a vital role in shaping the country's future.
'''

In [7]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign the summary. ${essay}'
structure_model.invoke(prompt).score

7

In [12]:
class EssayState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    indivisual_scores: Annotated[list[int],operator.add]
    # list e indivisual 3 ta number dibe - agulo k merge korar jonno add operator use krbe jeta reduce function
    # reducer function  - operator.add
    avg_score:float


In [ ]:
def evaluate_language(state:EssayState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign the summary. ${state['essay']}'
    output = structure_model.invoke(prompt)
    return {"language_feedback":output.feedback,'indivisual_scores':[output.score]}


In [ ]:
def evaluate_analysis(state:EssayState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign the summary. ${state['essay']}'
    output = structure_model.invoke(prompt)
    return {"analysis_feedback":output.feedback,'indivisual_scores':[output.score]}


In [25]:
def evaluate_thought(state:EssayState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign the summary. ${state['essay']}'
    output = structure_model.invoke(prompt)
    return {"clarity_feedback":output.feedback,'indivisual_scores':[output.score]}


In [26]:
def final_evaluation(state:EssayState):
    # summary feedback
    prompt = f'based on the folowing feedback  create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of asnalysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']} '
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['indivisual_scores']) / len(state['indivisual_scores'])
     
    return {'overall_feedback':overall_feedback, "avg_score":avg_score}

In [27]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

# edges
graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')
graph.add_edge('final_evaluation',END)

workflow = graph.compile()




In [28]:
initial_state = {
    'essay':essay
}
workflow.invoke(initial_state)

{'essay': "**The Rise of AI in Bangladesh**\n\nArtificial Intelligence (AI) is rapidly transforming Bangladesh and creating new opportunities in various sectors. With the growth of digital technology, businesses, educational institutions, and government organizations are increasingly adopting AI-based solutions to improve efficiency and productivity.\n\nIn the business sector, AI is being used for customer service, data analysis, fraud detection, and automation. Many startups and technology companies are developing AI-powered applications to solve local challenges. The banking and financial sectors are also using AI to enhance security and detect suspicious transactions.\n\nEducation is another area where AI is making a significant impact. Students now have access to intelligent learning tools, online tutors, and personalized educational resources. Universities are introducing courses related to AI, machine learning, and data science to prepare the next generation of professionals.\n\n